<a href="https://colab.research.google.com/github/phongsitt9/motor-glm-pricing/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🎯**โปรเจคเป็นการฝึกเรียนรู้การเป็น Actaury(pricing)**

# 🚗 01 — Data Preparation
## Motor GLM Pricing Project

In [1]:
# Dataset: freMTPL2 (French Motor Third-Party Liability)
from sklearn.datasets import fetch_openml

นี่คือ dataset มาตรฐานที่ใช้กันใน actuarial community ทั่วโลก มีข้อมูล 677,991 กรมธรรม์ motor insurance ครอบคลุมช่วงปี 2011–2013 แยกเป็น 2 ไฟล์
- freMTPL2freq (risk features + claim count)
- freMTPL2sev (claim amounts)





In [2]:
freq = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame

/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_openml.py:110: UserWarning: A network error occurred while downloading https://api.openml.org/api/v1/json/data/41214. Retrying...
  warn(


In [3]:
sev = fetch_openml(data_id=41215, as_frame=True, parser="auto").frame

In [8]:
print(f"Frequency Table : {freq.shape[0]:,} row , {freq.shape[1]} colums")
print(f"Severity Table : {sev.shape[0]:,} row , {sev.shape[1]} colums")

Frequency Table : 678,013 row , 12 colums
Severity Table : 26,639 row , 2 colums


In [10]:
freq.head(5)

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,'Regular',1217,R82
1,3.0,1,0.77,D,5,0,55,50,B12,'Regular',1217,R82
2,5.0,1,0.75,B,6,2,52,50,B12,'Diesel',54,R22
3,10.0,1,0.09,B,7,0,46,50,B12,'Diesel',76,R72
4,11.0,1,0.84,B,7,0,46,50,B12,'Diesel',76,R72


**Frequency table (freq) — ข้อมูลกรมธรรม์**
* IDpolรหัส policy (กรมธรรม์)
* จำนวน claim ที่เกิดขึ้น
* Exposureระยะเวลาคุ้มครอง (หน่วย: ปี)
* Areaความหนาแน่นของพื้นที่ที่อยู่
* VehPower
* VehAgeอายุรถ (ปี)0 = รถใหม่
* DrivAgeอายุผู้ขับ (ปี)46, 52
* BonusMalusคะแนน
* bonus-malus50 = baseline, >100 = ประวัติแย่
* VehBrandยี่ห้อรถB12, B1, ...
* VehGasประเภทเชื้อเพลิง Regular,Diesel
* Densityความหนาแน่นประชากรในพื้นที่ตัวเลขสูง =เมืองใหญ่
* Regionรหัสภูมิภาคในฝรั่งเศสR82, R22, R72


In [11]:
sev.head(5)

,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


**Severity table**
* IDpolรหัส policy
* ClaimAmountมูลค่า claim

In [13]:
sev_agg = (sev.groupby("IDpol")["ClaimAmount"]
              .sum()
              .reset_index())

In [15]:
num_duplicates = sev_agg.duplicated(subset=['IDpol']).sum()
print(f"จำนวน IDpol ที่ซ้ำใน sev_agg: {num_duplicates}")

if num_duplicates == 0:
    print("✅ ยืนยัน: IDpol ใน sev_agg ไม่มีการซ้ำกันแล้ว")
else:
    print("⚠️ คำเตือน: พบ IDpol ซ้ำใน sev_agg")

จำนวน IDpol ที่ซ้ำใน sev_agg: 0
✅ ยืนยัน: IDpol ใน sev_agg ไม่มีการซ้ำกันแล้ว


In [16]:
df = freq.merge(sev_agg, on="IDpol", how="left")

In [17]:
df["ClaimAmount"] = df["ClaimAmount"].fillna(0) # เติมตัว O ในช่อง NUN

In [18]:
print(f"✅ Merged table: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"   Policies with claims:    {(df['ClaimAmount'] > 0).sum():,}")
print(f"   Policies without claims: {(df['ClaimAmount'] == 0).sum():,}")

✅ Merged table: 678,013 rows, 13 columns
   Policies with claims:    24,944
   Policies without claims: 653,069


In [19]:
print(df.isnull().sum())

IDpol          0
ClaimNb        0
Exposure       0
Area           0
VehPower       0
VehAge         0
DrivAge        0
BonusMalus     0
VehBrand       0
VehGas         0
Density        0
Region         0
ClaimAmount    0
dtype: int64


In [21]:
print(df.dtypes)

IDpol           float64
ClaimNb           int64
Exposure        float64
Area           category
VehPower          int64
VehAge            int64
DrivAge           int64
BonusMalus        int64
VehBrand       category
VehGas           object
Density           int64
Region         category
ClaimAmount     float64
dtype: object


In [24]:
df[["Exposure", "ClaimNb", "ClaimAmount"]].describe().round(4)

,Exposure,ClaimNb,ClaimAmount
count,678013.0000,678013.0000,6.780130e+05
mean,0.5288,0.0532,8.836000e+01
std,0.3644,0.2401,5.822454e+03
min,0.0027,0.0000,0.000000e+00
25%,0.1800,0.0000,0.000000e+00
50%,0.4900,0.0000,0.000000e+00
75%,0.9900,0.0000,0.000000e+00
max,2.0100,16.0000,4.075401e+06


In [27]:
df["Exposure"] = df["Exposure"].clip(upper=1)

In [32]:
# สร้าง pure premium column (loss per exposure year)
df["PurePremium"] = df["ClaimAmount"] / df["Exposure"]
df["PurePremium"]

,PurePremium
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
678008,0.0
678009,0.0
678010,0.0
678011,0.0


In [33]:
print(f"   Exposure range: {df['Exposure'].min():.4f} – {df['Exposure'].max():.4f} years")
print(f"   Max ClaimNb: {df['ClaimNb'].max()}")
print(f"   Total losses: {df['ClaimAmount'].sum():,.0f}")

   Exposure range: 0.0027 – 1.0000 years
   Max ClaimNb: 16
   Total losses: 59,909,216


In [34]:
df.to_csv("clean_data.csv", index=False)

print("✅ Saved: clean_data.csv")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {list(df.columns)}")

✅ Saved: clean_data.csv
   Rows: 678,013
   Columns: ['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region', 'ClaimAmount', 'PurePremium']
